In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Define Nifty 50 universe
nifty_stocks = [
    "RELIANCE.NS", "HDFCBANK.NS", "ICICIBANK.NS", "INFY.NS", "TCS.NS",
    "LT.NS", "AXISBANK.NS", "SBIN.NS", "ITC.NS", "KOTAKBANK.NS",
    "HINDUNILVR.NS", "BHARTIARTL.NS", "ASIANPAINT.NS", "BAJFINANCE.NS",
    "BAJAJFINSV.NS", "SUNPHARMA.NS", "MARUTI.NS", "NTPC.NS",
    "POWERGRID.NS", "ONGC.NS", "ULTRACEMCO.NS", "TITAN.NS"
]

# Download price data and calculate log returns
tickers = ["^NSEI"] + nifty_stocks
data = yf.download(tickers, start="2020-01-01", progress=False, auto_adjust=True)['Close']
returns = np.log(data / data.shift(1)).dropna()

In [ ]:
window = 252
results = []

for stock in nifty_stocks:
    betas, r2s = [], []

    # Iterate through rolling windows for statistical significance
    for i in range(window, len(returns)):
        y = returns[stock].iloc[i-window:i]
        x = sm.add_constant(returns["^NSEI"].iloc[i-window:i])

        model = sm.OLS(y, x).fit()
        betas.append(model.params["^NSEI"])
        r2s.append(model.rsquared)

    results.append({
        "Stock": stock,
        "R2": np.mean(r2s),
        "Beta": np.mean(betas),
        "Volatility": returns[stock].std(),
        # Contribution = Beta * Volatility
        "Contribution": np.mean(betas) * returns[stock].std()
    })

df_stats = pd.DataFrame(results)
df_stats.sort_values("R2", ascending=False).head(10)

In [ ]:
# Static weights from NSE (Approximate)
nse_weights = {
    "RELIANCE.NS": 0.105,
    "HDFCBANK.NS": 0.085,
    "ICICIBANK.NS": 0.072,
    "INFY.NS": 0.065,
    "TCS.NS": 0.060,
    "LT.NS": 0.045,
    "AXISBANK.NS": 0.030,
    "SBIN.NS": 0.028,
    "ITC.NS": 0.027,
    "KOTAKBANK.NS": 0.026,
    "BAJFINANCE.NS": 0.019,
    "BAJAJFINSV.NS": 0.008,
    "ONGC.NS": 0.011
}

# Merge and compare
comparison = df_stats.set_index("Stock")[["Contribution"]]
comparison["Index Weight"] = pd.Series(nse_weights)

# Filter for comparison and sort by Beta Contribution
comparison = comparison.dropna().sort_values("Contribution", ascending=False)
comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.plot(kind='bar', figsize=(12, 6))
plt.title("Nifty 50: Index Weight vs. Growth Contribution")
plt.ylabel("Value")
plt.show()